# Count Lake and Non-Lake Pixels in TIFF Mask Files

This notebook reads all `.tif` / `.tiff` mask files from a labels folder.

Mask assumption:
- `0` = non-lake pixel
- `1` = lake pixel

Final output:

`ratio = number_of_non_lake_pixels / number_of_lake_pixels`


## 1. Install required library

Run this only if `rasterio` is not already installed.

In [ ]:
# Uncomment and run if rasterio is not installed
# !pip install rasterio

## 2. Import libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

## 3. Set labels folder path

Replace the path below with your folder path containing TIFF mask files.

In [2]:
labels_folder = Path(r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\subset_750\labels")  # <-- change this

if not labels_folder.exists():
    raise FileNotFoundError(f"Labels folder not found: {labels_folder}")

tiff_files = sorted(list(labels_folder.glob("*.tif")) + list(labels_folder.glob("*.tiff")))

print(f"Found {len(tiff_files)} TIFF files")
if len(tiff_files) == 0:
    raise FileNotFoundError(f"No .tif or .tiff files found in: {labels_folder}")

Found 750 TIFF files


## 4. Count lake and non-lake pixels

This cell counts only pixel values exactly equal to `0` and `1`. It also reports unexpected pixel values, if any exist.

In [3]:
total_non_lake_pixels = 0
total_lake_pixels = 0
total_other_pixels = 0
file_stats = []

for tif_path in tiff_files:
    with rasterio.open(tif_path) as src:
        # Read first band. Most binary masks have one band.
        mask = src.read(1)

        # Count pixels
        non_lake_count = int(np.sum(mask == 0))
        lake_count = int(np.sum(mask == 1))
        other_count = int(mask.size - non_lake_count - lake_count)

        total_non_lake_pixels += non_lake_count
        total_lake_pixels += lake_count
        total_other_pixels += other_count

        file_stats.append({
            "file_name": tif_path.name,
            "height": src.height,
            "width": src.width,
            "total_pixels": int(mask.size),
            "non_lake_pixels_0": non_lake_count,
            "lake_pixels_1": lake_count,
            "other_pixels": other_count
        })

stats_df = pd.DataFrame(file_stats)
stats_df.head()

,file_name,height,width,total_pixels,non_lake_pixels_0,lake_pixels_1,other_pixels
0,image_10.tif,512,512,262144,253300,8844,0
1,image_100.tif,512,512,262144,253821,8323,0
2,image_1000.tif,512,512,262144,258476,3668,0
3,image_1001.tif,512,512,262144,240184,21960,0
4,image_1002.tif,512,512,262144,251646,10498,0


## 5. Show final ratio

In [4]:
print("===== Final Pixel Counts =====")
print(f"Total non-lake pixels value 0: {total_non_lake_pixels:,}")
print(f"Total lake pixels value 1:     {total_lake_pixels:,}")
print(f"Total unexpected pixels:       {total_other_pixels:,}")

if total_lake_pixels == 0:
    ratio = float("inf")
    print("\nRatio cannot be computed normally because lake pixels are 0.")
else:
    ratio = total_non_lake_pixels / total_lake_pixels
    print(f"\nRatio = non-lake pixels / lake pixels = {ratio:.6f}")

ratio

===== Final Pixel Counts =====
Total non-lake pixels value 0: 192,974,551
Total lake pixels value 1:     3,633,449
Total unexpected pixels:       0

Ratio = non-lake pixels / lake pixels = 53.110571


53.11057097540106

## 6. Optional: Save per-file statistics to CSV

In [ ]:
output_csv = labels_folder / "mask_pixel_counts.csv"
stats_df.to_csv(output_csv, index=False)
print(f"Saved per-file statistics to: {output_csv}")